[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ADRIANVM117/quant-research-notes/blob/factor-momentum-colab-v1/Momentum_systematic_strategies/02_factor_momentum/notebooks/02_factor_momentum_signal_test.ipynb)`n
> El enlace público funcionará cuando esta versión esté publicada en GitHub. En Colab, ejecuta todas las celdas: el bootstrap descarga y verifica los siete insumos Gold fijados a una revisión concreta, sin requerir claves ni notebooks previas.

In [ ]:
# Bootstrap reproducible para Google Colab. No modifica la ejecución local.
from datetime import date
from hashlib import sha256
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import urlopen
import os
import subprocess
import sys

try:
    import google.colab
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    REQUIRED_PACKAGES = {"numpy": "2.5.4", "polars": "1.44.2", "plotly": "6.2.0", "nbformat": "5.10.4"}
    installed = {}
    for package in REQUIRED_PACKAGES:
        try:
            installed[package] = version(package)
        except PackageNotFoundError:
            installed[package] = None
    if any(installed[name] != required for name, required in REQUIRED_PACKAGES.items()):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                               "numpy==2.5.4", "polars==1.44.2", "plotly==6.2.0", "nbformat==5.10.4"])

    import numpy as np
    import plotly
    import polars as pl

    if np.__version__ != "2.5.4" or pl.__version__ != "1.44.2" or plotly.__version__ != "6.2.0":
        raise RuntimeError("Las versiones instaladas no coinciden con el entorno reproducible fijado.")

    SOURCE_COMMIT = "a8ab4d6bf04184551a783925eec1a94db0a31dd9"
    RAW_ROOT = ("https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes/"
                f"{SOURCE_COMMIT}/Momentum_systematic_strategies/data/gold/daily_series/etf")
    INPUTS = {
        "SPY": "b0767b45c2d67dbe8d99e11e1603682fd28468c31de65d037a4c0a50fefa67e9",
        "QQQ": "69760c426b11238aca1f1200f1c44933938954115d944db3b5a3df77618475fc",
        "IWM": "7913707e8a33b60780a7fe93617eb8114ef71b0e66e6a3b0b4dbab752cf09523",
        "IWD": "d73576d8c83dcbe57392174cb341061321ab963cd4adbdd063f455342caea0e0",
        "IWF": "d0c4165277bc9167421dde15ef9ede395bc35d259326de22913a76ef5792314e",
        "XLE": "f98e7468b5aee758298229a14443818ea0148bc46b26042a02abe1dee9af7a5a",
        "XLF": "52612673ab44bb3e222d5ac208bd3b9dfd0871c2259ff9fb472c59d902f170e0",
    }
    PROJECT_ROOT = Path("/content/quant-research-notes/Momentum_systematic_strategies")
    GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"
    GOLD_ROOT.mkdir(parents=True, exist_ok=True)

    for symbol, expected_hash in INPUTS.items():
        destination = GOLD_ROOT / f"{symbol}.parquet"
        if destination.exists():
            observed_hash = sha256(destination.read_bytes()).hexdigest()
            if observed_hash != expected_hash:
                raise RuntimeError(f"{symbol}: existe un archivo local con SHA-256 distinto; no se reemplaza.")
        else:
            url = f"{RAW_ROOT}/{symbol}.parquet"
            try:
                with urlopen(url, timeout=60) as response:
                    if response.status != 200:
                        raise RuntimeError(f"{symbol}: HTTP {response.status} al descargar {url}")
                    payload = response.read()
            except (HTTPError, URLError, TimeoutError) as error:
                raise RuntimeError(f"{symbol}: no fue posible descargar el insumo fijado.") from error
            observed_hash = sha256(payload).hexdigest()
            if observed_hash != expected_hash:
                raise RuntimeError(f"{symbol}: SHA-256 remoto inesperado; se cancela la ejecución.")
            temporary = destination.with_suffix(".parquet.download")
            temporary.write_bytes(payload)
            temporary.replace(destination)

    required_columns = {"session_date", "adjusted_close"}
    expected_start, expected_end, expected_sessions = date(2018, 1, 2), date(2026, 11, 17), 2232
    calendars, COLAB_INPUT_AUDIT = [], []
    for symbol, expected_hash in INPUTS.items():
        path = GOLD_ROOT / f"{symbol}.parquet"
        frame = pl.read_parquet(path).with_columns(pl.col("session_date").cast(pl.Date))
        missing = required_columns.difference(frame.columns)
        if missing:
            raise RuntimeError(f"{symbol}: faltan columnas requeridas: {sorted(missing)}")
        scoped = frame.filter(pl.col("session_date").is_between(expected_start, expected_end, closed="both"))
        invalid = (scoped.height != expected_sessions or scoped.get_column("session_date").null_count()
                   or scoped.get_column("adjusted_close").null_count()
                   or scoped.get_column("session_date").is_duplicated().any()
                   or scoped.get_column("session_date").to_list() != sorted(scoped.get_column("session_date").to_list())
                   or (scoped.get_column("adjusted_close") <= 0).any()
                   or scoped.get_column("session_date").min() != expected_start
                   or scoped.get_column("session_date").max() != expected_end)
        if invalid:
            raise RuntimeError(f"{symbol}: cobertura Gold inválida para la ventana congelada.")
        calendars.append(set(scoped.get_column("session_date").to_list()))
        COLAB_INPUT_AUDIT.append({"symbol": symbol, "sha256": expected_hash, "rows": scoped.height})

    common_dates = set.intersection(*calendars)
    if len(common_dates) != expected_sessions or min(common_dates) != expected_start or max(common_dates) != expected_end:
        raise RuntimeError("La intersección de calendarios Gold no coincide con las 2,232 sesiones esperadas.")
    os.chdir(PROJECT_ROOT)
    print(f"Colab listo | commit: {SOURCE_COMMIT} | sesiones comunes: {len(common_dates)}")
    print(f"Gold verificado: {GOLD_ROOT}")
    print(f"Versiones: numpy={np.__version__}, polars={pl.__version__}, plotly={plotly.__version__}")
else:
    print("Ejecución local detectada: no se instalaron dependencias ni se descargaron datos.")


# 02 — Prueba de señales de momentum factorial

> Este laboratorio compara señales long/cash de momentum total, de mercado y sistemático para QQQ, XLE y XLF. Es una prueba económica descriptiva con ejecución idealizada al cierre.

## <b>Experimento</b>

El horizonte está congelado en 126 sesiones. Cada señal conocida al cierre de $t$ se aplica al retorno logarítmico realizado en $t+1$: $R_{t+1}^{strategy} = w_t r_{t+1}$. Las alternativas son Buy & Hold, TSMOM total, market momentum de SPY y systematic momentum del activo.

In [8]:
from datetime import date
from pathlib import Path

import numpy as np
from IPython.display import HTML, display
import plotly.graph_objects as go
import polars as pl

TARGETS = ("QQQ", "XLE", "XLF")
SYMBOLS = ("SPY", "QQQ", "IWM", "IWD", "IWF", "XLE", "XLF")
FACTOR_COLUMNS = ("market", "size", "growth_value")
RESEARCH_START = date(2018, 1, 2)
RESEARCH_END = date(2026, 11, 17)
OLS_WINDOW = 252
MOMENTUM_HORIZON = 126
ILLUSTRATIVE_COST_BPS = 10

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        project_root = candidate / "Momentum_systematic_strategies"
        if (project_root / "data" / "gold" / "daily_series" / "etf").is_dir():
            return project_root
        if (candidate / "data" / "gold" / "daily_series" / "etf").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró data/gold/daily_series/etf.")

GOLD_ROOT = find_project_root(Path.cwd()) / "data" / "gold" / "daily_series" / "etf"
frames = {}
for symbol in SYMBOLS:
    path = GOLD_ROOT / f"{symbol}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Gold no disponible para {symbol}: {path}")
    frames[symbol] = (
        pl.read_parquet(path)
        .with_columns(pl.col("session_date").cast(pl.Date))
        .filter(pl.col("session_date").is_between(RESEARCH_START, RESEARCH_END, closed="both"))
        .select("session_date", "adjusted_close")
        .sort("session_date")
    )

common_panel = frames["SPY"].rename({"session_date": "date", "adjusted_close": "SPY"})
for symbol in SYMBOLS[1:]:
    common_panel = common_panel.join(
        frames[symbol].rename({"session_date": "date", "adjusted_close": symbol}),
        on="date",
        how="inner",
    )

common_panel = common_panel.sort("date")
common_panel.select("date", *SYMBOLS).head()

date,SPY,QQQ,IWM,IWD,IWF,XLE,XLF
date,f64,f64,f64,f64,f64,f64,f64
2018-01-02,235.20385,149.695763,138.13772,103.928786,31.872467,25.380449,23.706931
2018-01-03,236.69154,151.150312,138.281333,104.31984,32.144381,25.760551,23.834297
2018-01-04,237.689168,151.414775,138.649342,104.719214,32.277993,25.916047,24.055063
2018-01-05,239.273121,152.93544,138.936568,105.068667,32.571003,25.905681,24.122992
2018-01-08,239.710677,153.530483,139.151987,105.235072,32.662422,26.061177,24.089027


## <b>Atribución sistemática</b>

> Se replica la convención de Block 05: el modelo completo incluye intercepto, MKT = SPY, SIZE = IWM − SPY y GV = IWF − IWD. Para atribuir el retorno de una fecha, los coeficientes se ajustan con las 252 sesiones anteriores.

Los proxies contemporáneos se conocen al cierre de la sesión atribuida. Esta etapa construye componentes para señales posteriores; no pretende anticipar el retorno antes de ese cierre.

In [9]:
returns_panel = common_panel.with_columns([
    pl.col(symbol).log().diff().alias(f"{symbol}_ret") for symbol in SYMBOLS
]).drop_nulls([f"{symbol}_ret" for symbol in SYMBOLS])

model_panel = (
    returns_panel
    .select("date", *[f"{asset}_ret" for asset in TARGETS], "SPY_ret", "IWM_ret", "IWF_ret", "IWD_ret")
    .with_columns(
        pl.col("SPY_ret").alias("market"),
        (pl.col("IWM_ret") - pl.col("SPY_ret")).alias("size"),
        (pl.col("IWF_ret") - pl.col("IWD_ret")).alias("growth_value"),
    )
    .select("date", *[f"{asset}_ret" for asset in TARGETS], *FACTOR_COLUMNS)
    .drop_nulls()
)

oos_dates = model_panel.get_column("date").to_list()
factor_matrix = model_panel.select(*FACTOR_COLUMNS).to_numpy()
attribution_rows = []

for asset in TARGETS:
    actual_returns = model_panel.get_column(f"{asset}_ret").to_numpy()
    for index in range(OLS_WINDOW, len(actual_returns)):
        history = factor_matrix[index - OLS_WINDOW:index]
        design = np.column_stack((np.ones(OLS_WINDOW), history))
        coefficients = np.linalg.lstsq(design, actual_returns[index - OLS_WINDOW:index], rcond=None)[0]
        systematic_return = coefficients[0] + factor_matrix[index] @ coefficients[1:]
        attribution_rows.append({
            "date": oos_dates[index],
            "beta_asof_date": oos_dates[index - 1],
            "asset": asset,
            "actual_return": actual_returns[index],
            "systematic_return": systematic_return,
            "residual_return": actual_returns[index] - systematic_return,
            "beta_market": coefficients[1],
            "beta_size": coefficients[2],
            "beta_growth_value": coefficients[3],
        })

attribution = pl.DataFrame(attribution_rows)
attribution_audit = (
    attribution
    .with_columns((pl.col("actual_return") - pl.col("systematic_return") - pl.col("residual_return")).abs().alias("reconstruction_error"))
    .group_by("asset")
    .agg(
        (pl.col("beta_asof_date") < pl.col("date")).all().alias("betas_precede_return"),
        pl.col("reconstruction_error").max().alias("max_abs_reconstruction_error"),
    )
    .with_columns(pl.col("max_abs_reconstruction_error").round(12))
)
attribution_audit

asset,betas_precede_return,max_abs_reconstruction_error
str,bool,f64
"""XLF""",true,0.0
"""XLE""",true,0.0
"""QQQ""",true,0.0


## <b>Señales a 126d</b>

El momentum total, de mercado y sistemático acumula sólo retornos disponibles hasta la fecha de señal. Las tres reglas son long/cash sin apalancamiento: se está long cuando el componente correspondiente es positivo y en cash en caso contrario.

In [10]:
attribution_with_market = attribution.join(
    model_panel.select("date", pl.col("market").alias("market_return")),
    on="date",
    how="left",
)
signal_rows = []

for asset in TARGETS:
    sample = attribution_with_market.filter(pl.col("asset") == asset).sort("date")
    dates = sample.get_column("date").to_list()
    total = sample.get_column("actual_return").to_numpy()
    systematic = sample.get_column("systematic_return").to_numpy()
    market = sample.get_column("market_return").to_numpy()
    for end in range(MOMENTUM_HORIZON - 1, sample.height):
        start = end - MOMENTUM_HORIZON + 1
        signal_rows.append({
            "signal_date": dates[end],
            "asset": asset,
            "total_momentum_126d": total[start:end + 1].sum(),
            "market_momentum_126d": market[start:end + 1].sum(),
            "systematic_momentum_126d": systematic[start:end + 1].sum(),
        })

signals = pl.DataFrame(signal_rows)
next_returns = model_panel.select(
    pl.col("date").alias("signal_date"),
    pl.col("date").shift(-1).alias("return_date"),
    *[pl.col(f"{asset}_ret").shift(-1).alias(f"{asset}_next_return") for asset in TARGETS],
)

signal_panel = signals.join(next_returns, on="signal_date", how="inner").drop_nulls()
signal_panel.select("signal_date", "return_date", "asset", "total_momentum_126d", "systematic_momentum_126d").head()

signal_date,return_date,asset,total_momentum_126d,systematic_momentum_126d
date,date,str,f64,f64
2019-07-05,2019-07-08,"""QQQ""",0.247122,0.261145
2019-07-08,2019-07-09,"""QQQ""",0.198189,0.211584
2019-07-09,2019-07-10,"""QQQ""",0.191349,0.204422
2019-07-10,2019-07-11,"""QQQ""",0.19226,0.1995
2019-07-11,2019-07-12,"""QQQ""",0.183416,0.196096


## <b>Retornos económicos</b>

> La señal de cada fila se forma al cierre de `signal_date` y se multiplica únicamente por el retorno de `return_date`, la sesión siguiente. Todas las alternativas usan la misma ventana efectiva por activo.

In [11]:
ALTERNATIVES = ("Buy & Hold", "TSMOM total", "Market momentum", "Systematic momentum")
strategy_rows = []

for asset in TARGETS:
    sample = signal_panel.filter(pl.col("asset") == asset).sort("signal_date")
    for row in sample.iter_rows(named=True):
        weights = {
            "Buy & Hold": 1.0,
            "TSMOM total": float(row["total_momentum_126d"] > 0),
            "Market momentum": float(row["market_momentum_126d"] > 0),
            "Systematic momentum": float(row["systematic_momentum_126d"] > 0),
        }
        for alternative, weight in weights.items():
            strategy_rows.append({
                "signal_date": row["signal_date"],
                "return_date": row["return_date"],
                "asset": asset,
                "alternative": alternative,
                "weight": weight,
                "next_return": row[f"{asset}_next_return"],
                "strategy_return": weight * row[f"{asset}_next_return"],
            })

strategy_returns = pl.DataFrame(strategy_rows)
strategy_returns.group_by("asset", "alternative").len().sort("asset", "alternative")

asset,alternative,len
str,str,u32
"""QQQ""","""Buy & Hold""",1853
"""QQQ""","""Market momentum""",1853
"""QQQ""","""Systematic momentum""",1853
"""QQQ""","""TSMOM total""",1853
"""XLE""","""Buy & Hold""",1853
…,…,…
"""XLE""","""TSMOM total""",1853
"""XLF""","""Buy & Hold""",1853
"""XLF""","""Market momentum""",1853


In [12]:
metric_rows = []
cost_rows = []

for asset in TARGETS:
    for alternative in ALTERNATIVES:
        sample = strategy_returns.filter((pl.col("asset") == asset) & (pl.col("alternative") == alternative)).sort("return_date")
        returns = sample.get_column("strategy_return").to_numpy()
        weights = sample.get_column("weight").to_numpy()
        wealth = np.exp(np.cumsum(returns))
        drawdown = wealth / np.maximum.accumulate(wealth) - 1
        turnover = np.abs(np.diff(weights)).sum()
        annual_return = np.exp(returns.mean() * 252) - 1
        annual_volatility = returns.std(ddof=1) * np.sqrt(252)
        sharpe = returns.mean() / returns.std(ddof=1) * np.sqrt(252) if returns.std(ddof=1) else np.nan
        net_returns = returns - np.r_[0, np.abs(np.diff(weights))] * ILLUSTRATIVE_COST_BPS / 10_000

        metric_rows.append({
            "asset": asset,
            "alternative": alternative,
            "annualized_return_pct": annual_return * 100,
            "annualized_volatility_pct": annual_volatility * 100,
            "sharpe": sharpe,
            "max_drawdown_pct": drawdown.min() * 100,
            "mean_exposure_pct": weights.mean() * 100,
            "position_changes": int(np.count_nonzero(np.diff(weights))),
        })
        cost_rows.append({
            "asset": asset,
            "alternative": alternative,
            "turnover_per_year": turnover / len(weights) * 252,
            "gross_annualized_return_pct": annual_return * 100,
            "annualized_return_at_10bps_pct": (np.exp(net_returns.mean() * 252) - 1) * 100,
        })

performance_summary = pl.DataFrame(metric_rows).with_columns(pl.all().exclude("asset", "alternative", "position_changes").round(3))
cost_sensitivity = pl.DataFrame(cost_rows).with_columns(pl.all().exclude("asset", "alternative").round(3))
performance_summary

asset,alternative,annualized_return_pct,annualized_volatility_pct,sharpe,max_drawdown_pct,mean_exposure_pct,position_changes
str,str,f64,f64,f64,f64,f64,i64
"""QQQ""","""Buy & Hold""",19.008,24.453,0.712,-35.123,100.0,0
"""QQQ""","""TSMOM total""",14.4,18.406,0.731,-32.41,81.759,29
"""QQQ""","""Market momentum""",10.033,17.922,0.533,-29.316,80.464,56
"""QQQ""","""Systematic momentum""",12.918,18.097,0.671,-29.068,79.115,33
"""XLE""","""Buy & Hold""",14.879,33.019,0.42,-60.946,100.0,0
…,…,…,…,…,…,…,…
"""XLE""","""Systematic momentum""",15.571,23.17,0.625,-27.491,71.344,62
"""XLF""","""Buy & Hold""",12.118,23.862,0.479,-42.832,100.0,0
"""XLF""","""TSMOM total""",9.257,15.348,0.577,-21.715,73.664,60


### <b>Coste ilustrativo de rotación</b>

La siguiente sensibilidad descuenta 10 bps por unidad de cambio long/cash. Es un supuesto ilustrativo para comparar fragilidad frente al turnover; no representa tarifas, spreads ni costes reales de IBKR.

In [13]:
cost_sensitivity

asset,alternative,turnover_per_year,gross_annualized_return_pct,annualized_return_at_10bps_pct
str,str,f64,f64,f64
"""QQQ""","""Buy & Hold""",0.0,19.008,19.008
"""QQQ""","""TSMOM total""",3.944,14.4,13.95
"""QQQ""","""Market momentum""",7.616,10.033,9.198
"""QQQ""","""Systematic momentum""",4.488,12.918,12.413
"""XLE""","""Buy & Hold""",0.0,14.879,14.879
…,…,…,…,…
"""XLE""","""Systematic momentum""",8.432,15.571,14.601
"""XLF""","""Buy & Hold""",0.0,12.118,12.118
"""XLF""","""TSMOM total""",8.16,9.257,8.369


In [14]:
figure = go.Figure()
for alternative in ALTERNATIVES:
    sample = strategy_returns.filter((pl.col("asset") == "QQQ") & (pl.col("alternative") == alternative)).sort("return_date")
    figure.add_scatter(
        x=sample["return_date"],
        y=np.exp(np.cumsum(sample["strategy_return"].to_numpy())),
        mode="lines",
        name=alternative,
    )

figure.update_layout(
    title="QQQ — comparación de señales de momentum a 126d",
    xaxis_title="fecha de retorno",
    yaxis_title="crecimiento de 1",
    template="plotly_white",
)
display(HTML(figure.to_html(include_plotlyjs="cdn", full_html=False)))

In [15]:
figure = go.Figure()
for alternative in ALTERNATIVES:
    sample = performance_summary.filter(pl.col("alternative") == alternative).sort("asset")
    figure.add_bar(x=sample["asset"], y=sample["annualized_return_pct"], name=alternative)

figure.update_layout(
    title="Retorno anualizado por activo y señal",
    xaxis_title="activo",
    yaxis_title="retorno anualizado (%)",
    barmode="group",
    template="plotly_white",
)
display(HTML(figure.to_html(include_plotlyjs="cdn", full_html=False)))

## <b>Subperiodos retrospectivos</b>

Los periodos 2019–2022 y 2023–2026 se presentan como descripciones retrospectivas. Ambos ya han sido examinados durante esta investigación; no son holdouts confirmatorios independientes.

In [16]:
period_rows = []
for asset in TARGETS:
    for alternative in ALTERNATIVES:
        sample = strategy_returns.filter((pl.col("asset") == asset) & (pl.col("alternative") == alternative))
        for label, start, end in (("2019–2022", date(2019, 1, 1), date(2022, 12, 31)), ("2023–2026", date(2023, 1, 1), RESEARCH_END)):
            period = sample.filter(pl.col("return_date").is_between(start, end, closed="both")).sort("return_date")
            returns = period.get_column("strategy_return").to_numpy()
            period_rows.append({
                "asset": asset,
                "period": label,
                "alternative": alternative,
                "observations": period.height,
                "annualized_return_pct": (np.exp(returns.mean() * 252) - 1) * 100,
                "sharpe": returns.mean() / returns.std(ddof=1) * np.sqrt(252) if returns.std(ddof=1) else np.nan,
                "mean_exposure_pct": period.get_column("weight").mean() * 100,
            })

period_comparison = pl.DataFrame(period_rows).with_columns(
    pl.all().exclude("asset", "period", "alternative", "observations").round(3)
)
period_comparison

asset,period,alternative,observations,annualized_return_pct,sharpe,mean_exposure_pct
str,str,str,i64,f64,f64,f64
"""QQQ""","""2019–2022""","""Buy & Hold""",880,10.714,0.361,100.0
"""QQQ""","""2023–2026""","""Buy & Hold""",973,27.044,1.169,100.0
"""QQQ""","""2019–2022""","""TSMOM total""",880,8.349,0.404,72.955
"""QQQ""","""2023–2026""","""TSMOM total""",973,20.163,1.08,89.723
"""QQQ""","""2019–2022""","""Market momentum""",880,2.749,0.15,68.864
…,…,…,…,…,…,…
"""XLF""","""2023–2026""","""TSMOM total""",973,8.831,0.604,79.753
"""XLF""","""2019–2022""","""Market momentum""",880,4.312,0.226,68.864
"""XLF""","""2023–2026""","""Market momentum""",973,12.082,0.809,90.956


## <b>Interpretación</b>

- El systematic momentum mejora claramente TSMOM total en XLE: 15.571% frente a 5.615% anualizado, Sharpe 0.625 frente a 0.249 y drawdown -27.491% frente a -37.283%.
- En QQQ el systematic momentum queda por debajo de TSMOM total (12.918% frente a 14.400%) y en XLF también (7.995% frente a 9.257%); no hay una ventaja uniforme entre activos.
- Frente al market momentum, el componente sistemático mejora QQQ y XLE; en XLF ofrece retorno anual similarmente menor, pero menor turnover y drawdown.
- XLE combina mayor exposición sistemática (71.3% frente a 67.4% de TSMOM) con una mejora de retorno desproporcionada; en XLF la mayor exposición sistemática no evita un retorno menor, por lo que la exposición no explica todo el resultado.
- Los subperiodos cambian la lectura: XLE conserva la ventaja sistemática, mientras que QQQ la pierde en 2023–2026 y XLF pasa de desfavorable a favorable. Ambos periodos son retrospectivos.

## <b>Conclusión</b>

- El systematic momentum es económicamente favorable para XLE, desfavorable frente a TSMOM total para QQQ y XLF, y por tanto no supera de forma general al baseline clásico.
- Frente al market momentum, aporta una mejora material en QQQ y XLE; en XLF la diferencia de retorno es pequeña y acompaña menor turnover y drawdown.
- Buy & Hold conserva mayor retorno en QQQ y XLF, mientras que la señal sistemática mejora de forma notable el perfil retorno/riesgo de XLE.
- La sensibilidad ilustrativa de 10 bps no cambia la conclusión: XLE conserva ventaja sistemática, pero no convierte el resultado en una rentabilidad operable confirmada.
- La dependencia por activo y subperiodo limita cualquier lectura de alpha o de estabilidad estructural.

La siguiente lectura debe centrarse en si cualquier diferencia observada persiste después de considerar exposición, turnover y estabilidad entre subperiodos.

## <b>XLE — Fuentes de rendimiento incremental</b>

> Esta sección mantiene todas las señales congeladas y descompone únicamente la diferencia económica entre Systematic Momentum y TSMOM total para XLE.

La contribución diaria es $Delta R_{t+1} = (w_{systematic,t} - w_{tsmom,t}) r_{t+1}$. Una señal se forma al cierre de `signal_date` y se aplica al retorno de `return_date`.

In [17]:
xle_tsmom = strategy_returns.filter((pl.col("asset") == "XLE") & (pl.col("alternative") == "TSMOM total")).select(
    "signal_date", "return_date", "next_return", pl.col("weight").alias("tsmom_weight")
)
xle_systematic = strategy_returns.filter((pl.col("asset") == "XLE") & (pl.col("alternative") == "Systematic momentum")).select(
    "signal_date", pl.col("weight").alias("systematic_weight")
)
xle_market = strategy_returns.filter((pl.col("asset") == "XLE") & (pl.col("alternative") == "Market momentum")).select(
    "signal_date", pl.col("weight").alias("market_weight")
)
xle_signals = xle_tsmom.join(xle_systematic, on="signal_date").join(xle_market, on="signal_date").sort("signal_date")

position_rows = []
for label, column in (("TSMOM total", "tsmom_weight"), ("Systematic momentum", "systematic_weight"), ("Market momentum", "market_weight")):
    weights = xle_signals.get_column(column).to_numpy()
    metric = performance_summary.filter((pl.col("asset") == "XLE") & (pl.col("alternative") == label)).row(named=True)
    position_rows.append({
        "strategy": label,
        "time_invested_pct": weights.mean() * 100,
        "position_changes_per_year": np.count_nonzero(np.diff(weights)) / len(weights) * 252,
        "disagreement_vs_tsmom_pct": (weights != xle_signals.get_column("tsmom_weight").to_numpy()).mean() * 100,
        "annualized_return_pct": metric["annualized_return_pct"],
        "sharpe": metric["sharpe"],
        "max_drawdown_pct": metric["max_drawdown_pct"],
    })

xle_position_comparison = pl.DataFrame(position_rows).with_columns(pl.all().exclude("strategy").round(3))
xle_position_comparison

strategy,time_invested_pct,position_changes_per_year,disagreement_vs_tsmom_pct,annualized_return_pct,sharpe,max_drawdown_pct
str,f64,f64,f64,f64,f64,f64
"""TSMOM total""",67.404,11.696,0.0,5.615,0.249,-37.283
"""Systematic momentum""",71.344,8.432,19.698,15.571,0.625,-27.491
"""Market momentum""",80.464,7.616,36.373,9.871,0.382,-62.0


In [18]:
xle_differential = (
    xle_signals
    .with_columns(
        (pl.col("systematic_weight") - pl.col("tsmom_weight")).alias("weight_difference"),
        ((pl.col("systematic_weight") - pl.col("tsmom_weight")) * pl.col("next_return")).alias("delta_log_return"),
    )
    .with_columns(
        pl.when((pl.col("systematic_weight") == 1) & (pl.col("tsmom_weight") == 0))
        .then(pl.lit("Systematic long / TSMOM cash"))
        .when((pl.col("systematic_weight") == 0) & (pl.col("tsmom_weight") == 1))
        .then(pl.lit("Systematic cash / TSMOM long"))
        .otherwise(pl.lit("Agreement"))
        .alias("decision"),
    )
)

total_delta = xle_differential.get_column("delta_log_return").sum()
decision_rows = []
for decision in ("Systematic long / TSMOM cash", "Systematic cash / TSMOM long", "Agreement"):
    sample = xle_differential.filter(pl.col("decision") == decision)
    contribution = sample.get_column("delta_log_return").sum()
    decision_rows.append({
        "scope": "decision",
        "period": decision,
        "sessions": sample.height,
        "cumulative_log_contribution_pct": contribution * 100,
        "share_of_total_pct": contribution / total_delta * 100 if total_delta else np.nan,
    })

for year in sorted({value.year for value in xle_differential.get_column("return_date").to_list()}):
    sample = xle_differential.filter(pl.col("return_date").dt.year() == year)
    contribution = sample.get_column("delta_log_return").sum()
    decision_rows.append({
        "scope": "year",
        "period": str(year),
        "sessions": sample.height,
        "cumulative_log_contribution_pct": contribution * 100,
        "share_of_total_pct": contribution / total_delta * 100 if total_delta else np.nan,
    })

for label, start, end in (("2019–2022", date(2019, 1, 1), date(2022, 12, 31)), ("2023–2026", date(2023, 1, 1), RESEARCH_END)):
    sample = xle_differential.filter(pl.col("return_date").is_between(start, end, closed="both"))
    contribution = sample.get_column("delta_log_return").sum()
    decision_rows.append({
        "scope": "subperiod",
        "period": label,
        "sessions": sample.height,
        "cumulative_log_contribution_pct": contribution * 100,
        "share_of_total_pct": contribution / total_delta * 100 if total_delta else np.nan,
    })

decision_reconstruction = sum(row["cumulative_log_contribution_pct"] for row in decision_rows[:2]) / 100
if not np.isclose(decision_reconstruction, total_delta):
    raise AssertionError("Las contribuciones de desacuerdo no reconstruyen el diferencial total.")
print(f"Decision contributions reconstruct total differential: {decision_reconstruction * 100:.3f}% log return.")

xle_differential_summary = pl.DataFrame(decision_rows).with_columns(
    pl.all().exclude("scope", "period", "sessions").round(3)
)
xle_differential_summary

Decision contributions reconstruct total differential: 66.242% log return.


scope,period,sessions,cumulative_log_contribution_pct,share_of_total_pct
str,str,i64,f64,f64
"""decision""","""Systematic long / TSMOM cash""",219,69.253,104.545
"""decision""","""Systematic cash / TSMOM long""",146,-3.011,-4.545
"""decision""","""Agreement""",1488,0.0,0.0
"""year""","""2019""",124,-0.547,-0.826
"""year""","""2020""",253,41.234,62.248
…,…,…,…,…
"""year""","""2024""",252,24.768,37.39
"""year""","""2025""",250,14.361,21.68
"""year""","""2026""",221,0.0,0.0


In [19]:
episode_rows = []
active = None
for row in xle_differential.iter_rows(named=True):
    if row["decision"] == "Agreement":
        if active is not None:
            episode_rows.append(active)
            active = None
        continue
    if active is None or active["decision"] != row["decision"]:
        if active is not None:
            episode_rows.append(active)
        active = {"decision": row["decision"], "start": row["return_date"], "end": row["return_date"], "sessions": 0, "contribution": 0.0}
    active["end"] = row["return_date"]
    active["sessions"] += 1
    active["contribution"] += row["delta_log_return"]
if active is not None:
    episode_rows.append(active)

largest_episodes = sorted(episode_rows, key=lambda item: abs(item["contribution"]), reverse=True)[:4]
relevant_episodes_text = "\n".join(
    f"{item['start']} to {item['end']} — {item['decision']}; {item['sessions']} sessions; {item['contribution'] * 100:.2f}% log contribution"
    for item in largest_episodes
)
top_ten_abs_share = sum(abs(row["delta_log_return"]) for row in sorted(xle_differential.iter_rows(named=True), key=lambda item: abs(item["delta_log_return"]), reverse=True)[:10]) / sum(abs(row["delta_log_return"]) for row in xle_differential.iter_rows(named=True)) * 100

figure = go.Figure()
figure.add_scatter(
    x=xle_differential["return_date"],
    y=np.cumsum(xle_differential["delta_log_return"].to_numpy()) * 100,
    mode="lines",
    name="Systematic − TSMOM",
)
figure.add_hline(y=0, line_dash="dash", line_color="gray")
figure.update_layout(
    title="XLE — retorno logarítmico incremental acumulado de Systematic Momentum",
    xaxis_title="fecha de retorno",
    yaxis_title="diferencia acumulada de retorno logarítmico (%)",
    template="plotly_white",
)
display(HTML(figure.to_html(include_plotlyjs="cdn", full_html=False)))
print(f"Top 10 absolute daily contributions: {top_ten_abs_share:.1f}% of total absolute differential movement.")
print("Mechanically selected disagreement episodes by absolute cumulative contribution:")
print(relevant_episodes_text)

Top 10 absolute daily contributions: 14.0% of total absolute differential movement.
Mechanically selected disagreement episodes by absolute cumulative contribution:
2020-11-03 to 2020-11-24 — Systematic long / TSMOM cash; 16 sessions; 30.55% log contribution
2021-09-22 to 2021-10-20 — Systematic cash / TSMOM long; 21 sessions; -21.13% log contribution
2023-04-26 to 2023-05-16 — Systematic long / TSMOM cash; 15 sessions; -9.51% log contribution
2024-09-09 to 2024-11-06 — Systematic long / TSMOM cash; 43 sessions; 9.23% log contribution


### <b>Interpretación económica</b>

- Las señales discrepan el 19.7% de las sesiones. Systematic Momentum permanece invertido el 71.3% del tiempo, frente a 67.4% para TSMOM, y cambia posición con menor frecuencia anual (8.4 frente a 11.7).
- El diferencial exacto es 66.24% de log-retornos: las entradas sistemáticas con TSMOM en cash aportan 69.25%, mientras que las salidas sistemáticas con TSMOM long restan 3.01%.
- La contribución se reparte entre ambos subperiodos (64.0% en 2019–2022 y 36.0% en 2023–2026), aunque el episodio positivo de noviembre de 2020 es relevante. Los diez días individuales mayores representan 14.0% del movimiento absoluto.
- Market Momentum obtiene 9.871% anualizado con 80.5% de exposición, por debajo del 15.571% de la señal sistemática con 71.3%; la ventaja no parece reducirse a seguir el signo de SPY.

### <b>Conclusión de XLE</b>

- La diferencia favorable procede principalmente de capturar subidas durante entradas adicionales de Systematic Momentum, no de evitar pérdidas al salir frente a TSMOM.
- La señal sistemática tiene sólo 3.9 puntos porcentuales más de exposición y menor frecuencia de cambios, por lo que la mejora no se explica sólo por permanecer más tiempo invertida.
- La ventaja supera a Market Momentum pese a menor exposición, lo que sugiere que SIZE/GV contribuyen a la decisión frente a una señal basada sólo en SPY.
- La contribución está repartida entre ambos subperiodos, pero incluye episodios relevantes; el mismo historial no puede tratarse como validación independiente.
- XLE es una candidata razonable para una evaluación separada con ejecución y costes más realistas, sin interpretar el resultado como alpha confirmado.

Si la ventaja se mantiene tras esta descomposición, será una candidata razonable para una evaluación separada con ejecución y costes más realistas.